Autointerp using open source reasoning models to annotate SAE features

In [ ]:
# Load model quantized
# generalize this function

import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_name_or_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B" # deepseek-ai/DeepSeek-R1-Distill-Qwen-1B

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(model_name_or_path, quantization_config=config, device_map="cuda:0")

In [ ]:
model.eval()
# get gemmas activation db
# extract texts and activation paterns
auto_interp_prompt = []

In [ ]:
import gc
from tiny_dashboard import OfflineFeatureCentricDashboard
from huggingface_hub import hf_hub_download
repo_id = "Butanium/max-activating-examples-gemma-2-2b-l13-mu4.1e-02-lr1e-04"

# @markdown # Download max activating examples database
examples_source = "chat data" # @param ["chat data", "web data", "both chat and web"]
# @markdown Use 20 for faster download
num_max_activating_examples = 20 # @param [20, 100] {type:"raw"}
match examples_source:
    case "chat data":
        name = "chat"
    case "web data":
        name = "base"
    case "both chat and web":
        name = "chat_base"
num = "_20" if num_max_activating_examples == 20 else ""
db_path = hf_hub_download(repo_id=repo_id, filename=f"{name}_examples{num}.db", repo_type="dataset")
gc.collect()

In [14]:
import gc
import sqlite3
import pandas as pd
import json
from huggingface_hub import hf_hub_download
from nnterp import load_model

# Download the max activating examples database
repo_id = "Butanium/max-activating-examples-gemma-2-2b-l13-mu4.1e-02-lr1e-04"
examples_source = "chat data"  # choose among: "chat data", "web data", "both chat and web"
num_max_activating_examples = 20  # use 20 for faster download

# Select filename based on examples_source
match examples_source:
    case "chat data":
        name = "chat"
    case "web data":
        name = "base"
    case "both chat and web":
        name = "chat_base"
num = "_20" if num_max_activating_examples == 20 else ""
db_path = hf_hub_download(
    repo_id=repo_id, 
    filename=f"{name}_examples{num}.db", 
    repo_type="dataset"
)
gc.collect()

# Load the Gemma model (which includes its tokenizer)
gemma_2_it = load_model("google/gemma-2-2b-it", device_map="cuda:0")

# (Optional) List available tables in the database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Tables in the database:")
for table in tables:
    print(table[0])
conn.close()

# Now, load data from the known table ("data_table")
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM data_table LIMIT 20", conn)
conn.close()
print("\nDataFrame preview:")
print(df.head(1))

# Decode the examples.
# Each row's "entries" column is assumed to hold a list of pairs:
# [activation_value, [token0, token1, token2, ...]]
print("\nDecoded examples:")
for idx, row in df.iterrows():
    entry = row["entries"]
    # If the entry is stored as a JSON-formatted string, load it.
    if isinstance(entry, str):
        entry = json.loads(entry)
    # Check if there is at least one activation example
    if len(entry) > 0:
        # For example, take the first activation example
        activation_value, token_list = entry[0], entry[1]
        print(activation_value)
        print(entry[1])
    #     # Use the tokenizer’s helper to join tokens into a string.
    #     # (You can also use " ".join(token_list) if the tokens are already strings.)
    #     text = gemma_2_it.tokenizer.convert_tokens_to_string(token_list)
    #     print(f"Example {idx} (activation: {activation_value}):\n{text}\n")
    # else:
    #     print(f"Example {idx}: No entries found.\n")


Tables in the database:
data_table

DataFrame preview:
   key                                            entries
0   55  [[30.882600784301758, ["<bos>", "<start_of_tur...

Decoded examples:
[30.882600784301758, ['<bos>', '<start_of_turn>', 'user', '\n', 'X', ' is', ' located', ' within', ' the', ' Flä', 'ming', ' hill', ' range', ' and', ' in', ' the', ' centre', ' of', ' the', ' High', ' Flä', 'ming', ' Nature', ' Park', '.', ' The', ' plains', ' north', ' of', ' the', ' town', ' are', ' home', ' to', ' one', ' of', ' the', ' few', ' great', ' bust', 'ard', ' populations', ' in', ' Germany', '.', ' ', '\n\n', 'What', ' is', ' X', '?', '<end_of_turn>', '\n', '<start_of_turn>', 'model', '\n', 'Based', ' on', ' the', ' information', ' provided', ',', ' it', ' sounds', ' like', ' X', ' refers', ' to', ' a', ' town', ' located', ' within', ' the', ' Flä', 'ming', ' hill', ' range', ' and', ' in', ' the', ' centre', ' of', ' the', ' High', ' Flä', 'ming', ' Nature', ' Park', ' in', ' German

In [ ]:
# Add detection and fuzzy metric
# add sampling, adding top activations and random ones
# 

In [ ]:
dashboard = OfflineFeatureCentricDashboard.from_db(db_path, gemma_2_it.tokenizer, column_name="entries")
dashboard.display()

In [ ]:
# TODO results need to be averaged
# create a callable function for generating responses, consider batching?
# for now maybe give full outputs in jsonl file as well to be analyzed
# Parse the output so it only contains the brief description, save it in a list to be called later

# Generalize this in a function so its easily callable
device = "cuda:0"
model_inputs = tokenizer(auto_interp_prompt, return_tensors="pt")
tokens = model_inputs["input_ids"].to(device)
attention_mask = model_inputs["attention_mask"].to(device)
    # Note the length of the input
input_length = tokens.shape[1]
generation_output = model.generate(
    tokens,
    attention_mask=attention_mask,
    max_new_tokens = 1000,
    # top_p = 0.9,
    temperature=0.6,
    # do_sample=True,
)
new_tokens = generation_output[0, input_length:].tolist()  # Get only the new token ids
# output = tokenizer.decode(generation_output[0])
output = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(f'\n {output}')

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



  Hi there! I'm looking at these sentences where certain words are highlighted in brackets. Let me try to figure out the pattern here.

First sentence: "and he was <<over the moon>> to find"  
Second sentence: "we'll be laughing <<till the cows come home>>! Pro"  
Third sentence: "thought Scotland was boring, but really there's more <<than meets the eye>>! I'd"  

Hmm, all the highlighted words seem to be describing emotions or states of being. "Over the moon" shows excitement, "till the cows come home" conveys something very positive, and "than meets the eye" indicates a surprising revelation. 

Each of these phrases is used to express a strong emotion or a significant change in perspective. They're all examples of idiomatic expressions that highlight a particular feeling or situation.

So, the common pattern here is that the highlighted words are expressions that emphasize strong emotions or surprising truths. They're all used to convey a sense of intensity or revelation in the sent

In [ ]:
# For each text prediction run a auto_interp_reconstruct x times to reconstruct the activations
# Compare each predictions by giving end result - avg(Correctly_predicted_tokens/Total_tokens), 
# Maybe discard failed generations if the regex finds nothing
# Get the best results + caption for each feature output in jsonl:
    # feature_id, caption, avg_score

In [ ]:
auto_interp_reconstruct_prompt = [
    """
    Your task is to analyze the texts later given. Output only all of text that you analyze and highlight words/tokens in brackets like so <<example text>> if they correspond to a certain feature.
    
    Feature to highlight: The pattern is that the highlighted words are idiomatic expressions emphasizing emotions or significant realizations.

    {{Texts to analyze}}
 
    Reason step by step. Output the texts analyzed only, highlight the feature tokens in brackets like so <<example text>>.
    "<think>\n
    """
]

In [ ]:
# Call generation function

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



  I'm looking at the first text: "and he was over the moon to find". The phrase "over the moon" is an idiom that means being extremely happy or excited. It's expressing a strong emotion, which fits the feature.

     Moving to the second text: "we'll be laughing till the cows come home! Pro". The phrase "till the cows come home" is a common idiom meaning something will happen very soon. It doesn't convey a strong emotion or realization, so it doesn't fit the feature.

     Finally, the third text: "thought Scotland was boring, but really there's more than meets the eye! I'd". The phrase "there's more than meets the eye" is an idiom indicating that something is not what it seems. This reflects a significant realization, aligning with the feature.

     So, I'll highlight "over the moon" and "there's more than meets the eye" as they are idiomatic expressions emphasizing emotions or realizations.
</think>

1. and he was <<over the moon>> to find
2. we'll be laughing till the cows come ho

In [ ]:
# Show a scatterplot showing on y (% of correctly guessed tokens), 
# hovering over features you could see their id
# on x axis, feature id?
# TOP activating strings? 

In [ ]:
# could show a scatterplot comapring qwen-1b to qwen-7b?

In [ ]:
# crerate a u-map of the features